# **EVALUASI KEANDALAN MODEL: CER & WER**
## Skenario *speaker-dependent* dan *speaker-independent*

Notebook ini menjawab rumusan masalah ke-2: **bagaimana performa layanan model
transkripsi ditinjau dari metrik keandalan model (CER dan WER) pada skenario
*speaker-dependent* dan *speaker-independent*?**

Dua skenario itu menjawab pertanyaan yang berbeda:

| skenario | penutur pada data uji | pertanyaan yang dijawab |
|---|---|---|
| *speaker-dependent* | pernah dilatih | seberapa baik model mentranskripsi ucapan **orang yang suaranya sudah dikenal** |
| *speaker-independent* | **tidak pernah** dilatih | seberapa baik model mentranskripsi ucapan **orang baru** — inilah kondisi saat aplikasi dipakai pedagang lain |

Selisih antara keduanya adalah ukuran seberapa besar model bergantung pada
suara tertentu. Selisih yang lebar berarti model menghafal penutur, bukan
mempelajari bahasanya.

**Keluaran notebook ini:**

1. Tabel CER & WER per skenario, lengkap dengan selang kepercayaan *bootstrap*.
2. Rincian jenis kesalahan (substitusi / penghapusan / penyisipan).
3. Tabel dan grafik CER per penutur.
4. Uji signifikansi selisih antar-skenario.
5. Berkas `transkripsi_<skenario>.csv` berisi hipotesis per ucapan — dipakai
   ulang oleh notebook `08_evaluasi_nlu.ipynb` sebagai masukan NLU.

## **Syarat menjalankan**

Satu hal yang sering membuat angka *speaker-independent* jadi tidak sah, jadi
ditulis paling depan:

> **Tiap skenario harus dievaluasi memakai checkpoint yang dilatih pada skema
> pembagian data yang sama.** Model yang saat latihan sudah melihat seluruh
> penutur tidak bisa dipakai untuk mengukur *speaker-independent*, karena
> penutur "baru" pada data ujinya sebenarnya bukan penutur baru.

Jadi notebook ini butuh **dua checkpoint**:

| skenario | dilatih dengan | berkas hasil latihan |
|---|---|---|
| *speaker-dependent* | `SPLIT_SCHEME = "A"` | `best_ctc_A.keras` / `finsense_ctc_v11.keras` |
| *speaker-independent* | `SPLIT_SCHEME = "B"` | `best_ctc_B.keras` |

Keduanya dihasilkan oleh notebook pelatihan (Skema A dan Skema B). Kalau salah
satu belum ada, notebook tetap berjalan untuk skenario yang tersedia dan
menyatakan dengan jelas skenario mana yang dilewati — bukan mengarang angkanya.

Selain itu diperlukan dataset `finsense_expanded` (`transcripts_expanded.csv` +
folder `audio/`). Evaluasi berjalan di CPU; GPU hanya mempercepat.

**Checkpoint mana yang cocok dengan dataset ini.** Tidak semua model FinSense
dilatih di atas berkas yang sama, dan hanya yang sekorpus dengan notebook ini
yang boleh dievaluasi di sini:

| checkpoint | korpus latih | boleh dipakai di notebook ini? |
|---|---|---|
| `finsense_ctc_v11.keras` (Skema A) | `finsense_expanded` — 5100 rekaman apa adanya | **ya**, dengan `SPLIT_SD` yang sesuai |
| `best_ctc_B.keras` (Skema B) | `finsense_expanded`, penutur uji dikunci | **ya**, untuk skenario *speaker-independent* |
| `finsense_ctc_v12.keras` (Tahap 2) | korpus Tahap 1 — potongan hasil *forced alignment* + augmentasi dari 2500 rekaman unik, mencakup **seluruh** 1200 rekaman manusia | **tidak** |

Model v12 tidak boleh dipakai di sini karena korpus latihnya diturunkan dari
seluruh rekaman manusia yang ada, termasuk rekaman yang di notebook ini
diperlakukan sebagai data uji. Pembagian train/val pada korpus itu dikelompokkan
per rekaman sumber, bukan per penutur, dan tidak menyisakan data uji sama
sekali. Mengevaluasinya dengan pembagian di sini akan menghasilkan CER yang
tampak sangat baik justru karena bocor. Sel pemuatan model memberi peringatan
bila checkpoint seperti itu terdeteksi.

# **IMPORT LIBRARY**

Tahap ini memuat pustaka untuk pemrosesan audio, inferensi model, perhitungan
metrik, serta visualisasi hasil.

In [ ]:
import os, re, csv, json, glob, time, random, collections, math

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

print('TensorFlow :', tf.__version__)
print('Keras      :', keras.__version__)
print('librosa    :', librosa.__version__)
gpu = tf.config.list_physical_devices('GPU')
print('Perangkat  :', [g.name for g in gpu] if gpu else 'CPU saja (evaluasi tetap jalan, hanya lebih lambat)')

**Insight:**

Seed diset di satu tempat untuk seluruh pustaka acak. Ini bukan formalitas:
pembagian data uji pada notebook ini dibangkitkan ulang memakai seed yang sama
dengan saat pelatihan, sehingga himpunan uji yang dievaluasi benar-benar
himpunan yang tidak pernah dilihat model. Seed yang berbeda akan mengacak ulang
pembagian dan diam-diam memasukkan data latih ke dalam data uji.

# **KONFIGURASI**

Tahap ini menetapkan lokasi dataset, lokasi checkpoint tiap skenario, parameter
pembagian data, serta folder keluaran. Lokasi dicari otomatis di beberapa tempat
yang lazim (Kaggle, folder proyek, folder dataset lokal) agar notebook tetap
berjalan tanpa penyuntingan saat berpindah lingkungan.

**Parameter pembagian data wajib sama persis dengan yang dipakai saat
pelatihan.** Nilai bawaan di bawah mengikuti notebook pelatihan Skema A/B.

In [ ]:
# ─── 1. Dataset ───────────────────────────────────────────────────
# Diisi manual bila lokasi dataset berbeda; selain itu biarkan None.
DATASET_DIR_MANUAL = None

KANDIDAT_DATASET = [
    DATASET_DIR_MANUAL,
    '/kaggle/input/finsense-updated-dataset',
    '/kaggle/input/datasets/ibrahimirfanulhaq/finsense-updated-dataset',
    os.path.expanduser('~/finsense_expanded'),
    '../data/raw/finsense_expanded',
    '../../data/finsense_expanded',
]


def cari_dataset(kandidat):
    """Kembalikan (transcripts_csv, audio_dir) dari lokasi pertama yang lengkap."""
    for akar in kandidat:
        if not akar or not os.path.isdir(akar):
            continue
        csv_hit = sorted(glob.glob(os.path.join(akar, '**', 'transcripts_expanded.csv'),
                                   recursive=True))
        if not csv_hit:
            continue
        # Folder audio kadang bersarang (audio/audio) tergantung cara dataset diunggah.
        audio_hit = [d for d in sorted(glob.glob(os.path.join(akar, '**', 'audio'), recursive=True))
                     if os.path.isdir(d) and glob.glob(os.path.join(d, '*.wav'))]
        if audio_hit:
            return csv_hit[0], audio_hit[0]

    # Jaring pengaman untuk Kaggle: slug dataset bisa berbeda dari tebakan di
    # atas, jadi cari berkas penandanya di mana pun di bawah /kaggle/input.
    for csv_path in sorted(glob.glob('/kaggle/input/**/transcripts_expanded.csv',
                                     recursive=True)):
        akar = os.path.dirname(csv_path)
        for kandidat_audio in (os.path.join(akar, 'audio'), akar):
            audio_hit = [d for d in sorted(glob.glob(os.path.join(kandidat_audio, '**', '*.wav'),
                                                     recursive=True))]
            if audio_hit:
                return csv_path, os.path.dirname(audio_hit[0])
    return None, None


TRANSCRIPTS, AUDIO_DIR = cari_dataset(KANDIDAT_DATASET)
assert TRANSCRIPTS and AUDIO_DIR, (
    'Dataset tidak ditemukan. Isi DATASET_DIR_MANUAL dengan folder yang memuat '
    'transcripts_expanded.csv dan folder audio/.')

# ─── 2. Checkpoint per skenario ───────────────────────────────────
# Kunci dict ini adalah nama skenario; nilainya daftar lokasi yang dicoba
# berurutan. Checkpoint PERTAMA yang ada yang dipakai.
# Pola dicoba BERURUTAN; yang pertama ditemukan dipakai. Nama yang eksplisit
# menyebut skemanya (best_ctc_A / _B) didahulukan, baru nama umum
# finsense_ctc_v11.keras — sebab notebook pelatihan Skema A maupun Skema B
# sama-sama mengekspor berkas dengan nama umum itu. Kalau kedua checkpoint
# diunggah dengan nama yang sama, pencarian tidak mungkin membedakannya:
# beri nama berbeda saat mengunggah, atau isi jalurnya langsung di sini.
POLA_MODEL = {
    'speaker_dependent': [
        '/kaggle/input/**/best_ctc_A.keras',
        '/kaggle/input/**/finsense_ctc_v11_A.keras',
        './best_ctc_A.keras',
    ],
    'speaker_independent': [
        '../../ai-service/model_v11_si/finsense_ctc_v11_si.keras',
        '/kaggle/input/**/best_ctc_B.keras',
        '/kaggle/input/**/finsense_ctc_v11_B.keras',
        '/kaggle/input/**/finsense_ctc_v11_si.keras',
        './best_ctc_B.keras',
    ],
}


def pilih_berkas(pola_list):
    for pola in pola_list:
        hit = sorted(glob.glob(pola, recursive=True))
        if hit:
            return hit[0]
    return None


MODEL_SKEMA = {k: pilih_berkas(v) for k, v in POLA_MODEL.items()}

# ── Jaring pengaman: kenali checkpoint dari POLANYA, bukan dari nama persis ──
# Daftar nama di atas tidak akan pernah lengkap — orang menamai berkasnya
# bermacam-macam (finsense_ctc_A.keras, model_skema_b.keras, ...). Bila sebuah
# skenario belum dapat checkpoint, seluruh berkas .keras yang terjangkau
# diperiksa dan digolongkan lewat penanda A/B pada namanya.
AKAR_PENCARIAN_MODEL = ['/kaggle/input', '/kaggle/working', '../../ai-service', '.']


def golongkan_checkpoint(path):
    """Tebak skenario dari nama berkas -> 'speaker_dependent' / 'speaker_independent' / None.

    Penanda B (independent) diperiksa lebih dulu: kata 'independent' memuat
    'dependent' di dalamnya, jadi urutan terbalik akan salah menggolongkan.
    """
    nama = os.path.basename(path).lower()
    if re.search(r'(^|[_\-])(b|si|indep|independen|independent)([_\-.]|$)', nama):
        return 'speaker_independent'
    if re.search(r'(^|[_\-])(a|sd|dep|dependen|dependent)([_\-.]|$)', nama):
        return 'speaker_dependent'
    return None


SEMUA_KERAS = []
for akar in AKAR_PENCARIAN_MODEL:
    if os.path.isdir(akar):
        SEMUA_KERAS += sorted(glob.glob(os.path.join(akar, '**', '*.keras'), recursive=True))

if not all(MODEL_SKEMA.values()):
    for berkas in SEMUA_KERAS:
        skema = golongkan_checkpoint(berkas)
        if skema and not MODEL_SKEMA.get(skema):
            MODEL_SKEMA[skema] = berkas
            print(f'[COCOK POLA] {skema} <- {berkas}')

# Upaya terakhir untuk skenario speaker-dependent: berkas bernama umum
# finsense_ctc_v11.keras. Sengaja dicoba paling akhir — berkas yang namanya
# jelas menyebut skema selalu didahulukan, sebab notebook Skema A dan Skema B
# sama-sama mengekspor nama umum itu dan isinya tidak dapat dibedakan.
if not MODEL_SKEMA.get('speaker_dependent'):
    umum = pilih_berkas(['../../ai-service/model_v11_ctc/finsense_ctc_v11.keras',
                         '/kaggle/input/**/finsense_ctc_v11.keras',
                         './finsense_ctc_v11.keras'])
    if umum:
        MODEL_SKEMA['speaker_dependent'] = umum
        print(f'[NAMA UMUM] speaker_dependent <- {umum}')
        print('  Pastikan berkas ini memang hasil pelatihan Skema A.')

VOCAB_PATH = pilih_berkas([
    '../../ai-service/model_v11_ctc/char_vocab.json',
    '/kaggle/input/**/char_vocab.json',
    './char_vocab.json',
])
assert VOCAB_PATH, 'char_vocab.json tidak ditemukan — wajib untuk men-decode keluaran model.'

# ─── 3. Parameter pembagian data (HARUS sama dengan saat pelatihan) ──
# Dua notebook pelatihan mendefinisikan "speaker-dependent" dengan cara berbeda,
# dan memilih yang keliru di sini membuat sebagian data uji pernah dilihat model:
#   'per_penutur' — rekaman TIAP penutur dibagi train/val/test
#                   (Skema A pada notebook pelatihan gabungan A/B)
#   'acak'        — train_test_split acak atas seluruh baris
#                   (notebook pelatihan "Scheme A")
# Isi sesuai cara checkpoint speaker-dependent Anda dilatih.
SPLIT_SD        = 'per_penutur'

TEST_FRAC       = 0.15
VAL_FRAC        = 0.15
N_SPEAKERS      = 20     # berkas 1..1200 = 20 penutur x 60 rekaman
FILES_PER_SPK   = 60
N_TEST_SPEAKERS = 3
N_VAL_SPEAKERS  = 3

# ─── 4. Korpus Tahap 1 (opsional) ─────────────────────────────────
# Korpus potongan hasil forced alignment + augmentasi (lihat data-science/tahap1/).
# Dipakai untuk menguji model yang BENAR-BENAR dilayankan aplikasi (v12), yang
# dilatih di atas korpus itu — bukan di atas finsense_expanded. Bila tidak ada,
# bagian evaluasi Tahap 1 dilewati dan sisa notebook tetap berjalan.
def cari_korpus_tahap1():
    kandidat = [
        os.path.expanduser('~/finsense_tahap1'),
        '../../korpus_tahap1',
        './korpus_tahap1',
    ]
    for d in kandidat:
        if d and os.path.exists(os.path.join(d, 'manifest_val.csv')):
            return d
    hit = sorted(glob.glob('/kaggle/input/**/manifest_val.csv', recursive=True))
    return os.path.dirname(hit[0]) if hit else None


KORPUS_TAHAP1 = cari_korpus_tahap1()
MODEL_TAHAP1 = pilih_berkas([
    '../../ai-service/model_v12/finsense_ctc_v12.keras',
    '/kaggle/input/**/finsense_ctc_v12.keras',
    '/kaggle/input/**/*v12*.keras',
    '/kaggle/input/**/*tahap2*.keras',
    './finsense_ctc_v12.keras',
])

# ─── 5. Pelaksanaan evaluasi ──────────────────────────────────────
BATCH_INFER  = 16      # ukuran batch saat prediksi
MAKS_SAMPEL  = None    # None = seluruh data uji; isi angka untuk uji cepat
N_BOOTSTRAP  = 2000    # ulangan bootstrap untuk selang kepercayaan
N_LATENSI    = 50      # jumlah klip untuk pengukuran latensi per ucapan

KELUARAN = '/kaggle/working/hasil_evaluasi_asr' if os.path.isdir('/kaggle/working') else '../hasil/evaluasi_asr'
os.makedirs(KELUARAN, exist_ok=True)

print('skema SD   :', SPLIT_SD)
print('transkrip  :', TRANSCRIPTS)
print('audio      :', AUDIO_DIR)
print('vocab      :', VOCAB_PATH)
print('keluaran   :', os.path.abspath(KELUARAN))
print()
for skema, path in MODEL_SKEMA.items():
    print(f'{skema:22s}: {path or "TIDAK DITEMUKAN — skenario ini akan dilewati"}')
print()
print('korpus Tahap 1        :', KORPUS_TAHAP1 or 'tidak ada — evaluasi Tahap 1 dilewati')
print('model korpus Tahap 1  :', MODEL_TAHAP1 or 'tidak ada — evaluasi Tahap 1 dilewati')

if not all(MODEL_SKEMA.values()):
    print('\nBerkas .keras yang terlihat dari notebook ini:')
    for berkas in SEMUA_KERAS or ['(tidak ada satu pun)']:
        print('   ', berkas)
    print('Bila checkpoint Anda ada di daftar itu tetapi tidak terpasang ke skenario,')
    print('isi POLA_MODEL di atas dengan jalur berkasnya secara langsung.')

**Insight:**

Checkpoint dicari terpisah untuk tiap skenario, bukan satu model untuk keduanya.
Alasannya dijelaskan di bagian syarat: model yang saat pelatihan sudah melihat
seluruh penutur tidak dapat mengukur kondisi *speaker-independent*.

Bila checkpoint Skema B belum ada, keluaran di atas akan menyatakannya dan
notebook hanya melaporkan skenario yang datanya sah. Melewati satu skenario
lebih jujur daripada melaporkan angka yang tidak berarti.

# **LOAD DATA**

Tahap ini memuat daftar transkrip, menormalkan teksnya, lalu memetakan tiap
rekaman ke penuturnya. Normalisasi teks dibuat identik dengan tahap pelatihan
(huruf kecil, hanya `a-z` dan spasi), sebab acuan CER/WER harus berada di ruang
simbol yang sama dengan keluaran model.

In [ ]:
def bersihkan_teks(t):
    """Normalisasi identik dengan tahap pelatihan: huruf kecil, hanya a-z dan spasi."""
    t = str(t).lower().strip()
    t = re.sub(r'[^a-z ]', '', t)
    return re.sub(r'\s+', ' ', t)


df = pd.read_csv(TRANSCRIPTS)
df['teks'] = df['transcript'].apply(bersihkan_teks)


def id_penutur(filename):
    """Berkas bernomor 1..1200 = rekaman manusia, 60 berkas per penutur.

    Berkas lain (gen_*.wav dan turunan augmentasinya) adalah ucapan sintetis dan
    diberi id -1: penuturnya bukan orang yang dapat dihitung sebagai penutur.
    """
    m = re.match(r'^(\d+)\.wav$', str(filename))
    if m:
        n = int(m.group(1))
        if 1 <= n <= N_SPEAKERS * FILES_PER_SPK:
            return (n - 1) // FILES_PER_SPK
    return -1


df['penutur'] = df['filename'].apply(id_penutur)

n_manusia = int((df['penutur'] >= 0).sum())
n_sintetis = int((df['penutur'] == -1).sum())
n_penutur = df.loc[df.penutur >= 0, 'penutur'].nunique()

print(f'total ucapan   : {len(df):,}')
print(f'rekaman manusia: {n_manusia:,} dari {n_penutur} penutur')
print(f'ucapan sintetis: {n_sintetis:,}')
print(f'panjang teks   : min {df.teks.str.len().min()} | median {int(df.teks.str.len().median())} | maks {df.teks.str.len().max()} karakter')
assert n_penutur == N_SPEAKERS, f'Ditemukan {n_penutur} penutur, seharusnya {N_SPEAKERS}'
df.head()

**Insight:**

Identitas penutur tidak tersimpan sebagai kolom di dataset, melainkan tersirat
pada penomoran berkas: 60 rekaman berurutan berasal dari satu orang. Pemetaan
itulah yang memungkinkan pembagian *speaker-independent* dibuat sama sekali.

Ucapan sintetis (`gen_*`) sengaja tidak diberi identitas penutur. Ia boleh
dipakai melatih, tetapi tidak boleh dihitung sebagai "penutur baru" pada
pengujian — kalau ikut masuk data uji *speaker-independent*, yang terukur bukan
lagi kemampuan model menghadapi orang baru.

# **PEMBAGIAN DATA: SPEAKER-DEPENDENT vs SPEAKER-INDEPENDENT**

Tahap ini membangun ulang dua skema pembagian data yang dipakai saat pelatihan.
Fungsinya disalin apa adanya dari notebook pelatihan — termasuk urutan
pemanggilan bilangan acaknya — sehingga dengan seed yang sama himpunan ujinya
identik dengan yang dipakai saat model dilatih.

**Skema A (*speaker-dependent*).** Rekaman tiap penutur dibagi ke train, val,
dan test. Setiap penutur muncul di ketiga bagian, jadi saat diuji model sudah
pernah mendengar suara orang tersebut — hanya kalimatnya yang berbeda.

Ada dua cara Skema A ditulis pada notebook pelatihan, dan keduanya disediakan di
sini lewat `SPLIT_SD`: `'per_penutur'` (rekaman tiap penutur dibagi tiga) dan
`'acak'` (pembagian acak atas seluruh baris, seperti notebook pelatihan
"Scheme A"). Keduanya sama-sama *speaker-dependent*, tetapi **himpunan ujinya
berbeda**. Pilih yang sama dengan cara checkpoint dilatih; bila keliru, sebagian
ucapan yang di sini disebut "data uji" sebenarnya pernah dipakai melatih, dan
CER-nya menjadi terlalu bagus tanpa ada yang terlihat salah.

**Skema B (*speaker-independent*).** Yang dibagi adalah **penuturnya**, bukan
rekamannya: 3 penutur dikunci untuk uji, 3 untuk validasi, 14 sisanya untuk
latih. Ucapan sintetis hanya boleh masuk data latih, sehingga data uji 100%
manusia dan 100% suara baru.

In [ ]:
def split_speaker_dependent(data, seed=SEED):
    """Skema A: tiap penutur dibagi ke train/val/test. Sintetis dibagi acak."""
    rng = np.random.RandomState(seed)
    tr, va, te = [], [], []
    for spk in sorted(data[data.penutur >= 0]['penutur'].unique()):
        idx = np.array(data[data.penutur == spk].index.tolist())
        rng.shuffle(idx)
        n = len(idx); n_te = int(n * TEST_FRAC); n_va = int(n * VAL_FRAC)
        te += list(idx[:n_te])
        va += list(idx[n_te:n_te + n_va])
        tr += list(idx[n_te + n_va:])
    sintetis = np.array(data[data.penutur == -1].index.tolist())
    if len(sintetis):
        rng.shuffle(sintetis)
        n = len(sintetis); n_te = int(n * TEST_FRAC); n_va = int(n * VAL_FRAC)
        te += list(sintetis[:n_te])
        va += list(sintetis[n_te:n_te + n_va])
        tr += list(sintetis[n_te + n_va:])
    return sorted(tr), sorted(va), sorted(te)


def split_acak(data, seed=SEED):
    """Skema A versi notebook "Scheme A": pembagian acak atas seluruh baris.

    Urutan pemanggilan train_test_split dipertahankan persis seperti notebook
    pelatihan, sebab itulah yang menentukan baris mana yang menjadi data uji.
    """
    idx = np.arange(len(data))
    idx_tv, idx_test = train_test_split(idx, test_size=TEST_FRAC, random_state=seed)
    idx_train, idx_val = train_test_split(
        idx_tv, test_size=VAL_FRAC / (1 - TEST_FRAC), random_state=seed)
    return sorted(idx_train.tolist()), sorted(idx_val.tolist()), sorted(idx_test.tolist())


def split_speaker_independent(data, seed=SEED):
    """Skema B: penutur uji/validasi tidak pernah muncul saat latih.

    Ucapan sintetis hanya masuk data latih supaya data uji benar-benar berisi
    suara manusia yang baru bagi model.
    """
    rng = np.random.RandomState(seed)
    sp = list(range(N_SPEAKERS)); rng.shuffle(sp)
    te_s = set(sp[:N_TEST_SPEAKERS])
    va_s = set(sp[N_TEST_SPEAKERS:N_TEST_SPEAKERS + N_VAL_SPEAKERS])
    tr_s = set(sp[N_TEST_SPEAKERS + N_VAL_SPEAKERS:])
    tr = data[data.penutur.isin(tr_s)].index.tolist()
    va = data[data.penutur.isin(va_s)].index.tolist()
    te = data[data.penutur.isin(te_s)].index.tolist()
    tr += data[data.penutur == -1].index.tolist()
    print(f'  penutur latih : {sorted(tr_s)}')
    print(f'  penutur valid : {sorted(va_s)}')
    print(f'  penutur uji   : {sorted(te_s)}')
    return sorted(tr), sorted(va), sorted(te)


SPLIT = {}
print(f'=== Skema A — speaker-dependent (pembagian: {SPLIT_SD}) ===')
SPLIT['speaker_dependent'] = (split_acak(df) if SPLIT_SD == 'acak'
                              else split_speaker_dependent(df))
print('\n=== Skema B — speaker-independent ===')
SPLIT['speaker_independent'] = split_speaker_independent(df)

ringkas = []
for skema, (tr, va, te) in SPLIT.items():
    spk_tr = set(df.loc[tr, 'penutur']) - {-1}
    spk_te = set(df.loc[te, 'penutur']) - {-1}
    ringkas.append({
        'skema': skema,
        'n_latih': len(tr), 'n_validasi': len(va), 'n_uji': len(te),
        'penutur_latih': len(spk_tr), 'penutur_uji': len(spk_te),
        'penutur_uji_baru': len(spk_te - spk_tr),
        'sintetis_di_uji': int((df.loc[te, 'penutur'] == -1).sum()),
    })
tabel_split = pd.DataFrame(ringkas)

# Verifikasi wajib: skema B tidak boleh bocor penutur maupun kebawa sintetis.
tr_b, va_b, te_b = SPLIT['speaker_independent']
spk = lambda ids: set(df.loc[ids, 'penutur']) - {-1}
assert not (spk(tr_b) & spk(te_b)), 'BOCOR: penutur latih muncul di data uji Skema B'
assert not (spk(va_b) & spk(te_b)), 'BOCOR: penutur validasi muncul di data uji Skema B'
assert (df.loc[te_b, 'penutur'] == -1).sum() == 0, 'Data uji Skema B berisi ucapan sintetis'
print('\nVerifikasi Skema B lolos: data uji 100% penutur baru, tanpa ucapan sintetis.\n')
tabel_split

**Insight:**

Perhatikan kolom `penutur_uji_baru`. Pada Skema A nilainya 0 — seluruh penutur
di data uji juga ada di data latih, dan memang itulah definisi
*speaker-dependent*. Pada Skema B nilainya sama dengan `penutur_uji`, artinya
tidak satu pun penutur uji pernah didengar model.

Kolom `sintetis_di_uji` menegaskan perbedaan berikutnya: data uji Skema B hanya
berisi rekaman manusia. Angka CER/WER Skema B karena itu tidak bisa terbantu
oleh ucapan sintetis yang pola akustiknya lebih mudah.

# **PEMERIKSAAN KEBOCORAN DATA**

Sebelum angka apa pun dihitung, data uji diperiksa dari dua jenis kebocoran yang
akan membuat CER terlihat lebih bagus daripada kenyataannya:

1. **Kebocoran penutur** — penutur yang sama muncul di latih dan uji. Wajar pada
   Skema A, tidak boleh terjadi pada Skema B.
2. **Kebocoran varian augmentasi** — satu rekaman sumber dipanen menjadi
   beberapa berkas (`gen_00165.wav`, `gen_00165_p+2.wav`, `gen_00165_nz15.wav`).
   Kalau sebagiannya masuk latih dan sisanya masuk uji, model sebenarnya sedang
   diuji pada rekaman yang pernah dilihatnya dalam bentuk lain.

Pemeriksaan ini tidak mengubah pembagian data — mengubahnya justru akan membuat
angka tidak lagi sepadan dengan checkpoint yang dilatih. Yang dilakukan adalah
**melaporkan** besarnya, dan menyediakan subset bersih sebagai pembanding.

In [ ]:
def id_sumber(filename):
    """Identitas rekaman sumber, lepas dari label augmentasinya.

    gen_00165.wav, gen_00165_p+2.wav, gen_00165_nz15.wav -> gen_00165
    """
    nama = os.path.splitext(str(filename))[0]
    return re.sub(r'_(p[+-]\d+|nz\d+|sp\d+)$', '', nama)


df['sumber'] = df['filename'].apply(id_sumber)

baris = []
for skema, (tr, va, te) in SPLIT.items():
    sumber_latih = set(df.loc[tr, 'sumber'])
    bocor = df.loc[te, 'sumber'].isin(sumber_latih)
    spk_tr = set(df.loc[tr, 'penutur']) - {-1}
    spk_te = set(df.loc[te, 'penutur']) - {-1}
    baris.append({
        'skema': skema,
        'penutur_bocor': len(spk_tr & spk_te),
        'ucapan_uji': len(te),
        'varian_augmentasi_bocor': int(bocor.sum()),
        'persen_bocor': round(100 * bocor.mean(), 1),
    })
    df.loc[te, 'bocor_' + skema] = bocor.values

tabel_bocor = pd.DataFrame(baris)
tabel_bocor.to_csv(os.path.join(KELUARAN, 'pemeriksaan_kebocoran.csv'), index=False)
tabel_bocor

**Insight:**

Kebocoran penutur pada Skema A memang disengaja — itulah yang sedang diukur.
Yang perlu diperhatikan adalah kolom `varian_augmentasi_bocor`: pada Skema A,
ucapan sintetis dibagi secara acak per berkas, sehingga varian augmentasi dari
satu rekaman sumber dapat berada di kedua sisi. Untuk bagian itu model diuji
pada rekaman yang pernah dilihatnya dalam versi lain, dan CER-nya menjadi
optimistis.

Pada Skema B kolom tersebut nol: data ujinya hanya rekaman manusia dari penutur
yang seluruh rekamannya dikunci di sisi uji. Karena itu Skema B bukan sekadar
"skenario lain", melainkan angka yang lebih pantas dikutip sebagai perkiraan
kinerja di lapangan.

Bagian hasil nanti melaporkan CER pada subset bersih (tanpa kebocoran varian)
secara terpisah, sehingga besarnya pengaruh hal ini terlihat, bukan ditebak.

# **MUAT MODEL**

Tahap ini memuat checkpoint tiap skenario. Ada dua hal yang harus ditangani
sebelum berkasnya bisa dipakai.

**Lapisan kustom.** Berkas `.keras` menyimpan nama kelas lapisannya, bukan
kodenya, jadi tiap lapisan kustom harus didaftarkan ulang di sini — kalau tidak,
pemuatan gagal dengan `TypeError`. Yang dipakai kedua notebook pelatihan:
`SpecAugment`, `VTLPAugment`, dan `CTCLossLayer`. Semuanya didaftarkan sebagai
*pass-through*: augmentasi wajib mati saat mengukur, dan nilai loss tidak
dibutuhkan untuk menghitung CER/WER.

**Bentuk checkpoint.** Notebook pelatihan menghasilkan dua macam berkas:

| berkas | ukuran | masukan | isi |
|---|---|---|---|
| `best_ctc_*.keras` / `finsense_ctc_A.keras` | ~24 MB | 4 (log_mel, labels, input_len, label_len) | checkpoint pelatihan, lengkap dengan jalur loss |
| `finsense_ctc_v11.keras` | ~8 MB | 1 (log_mel) | model inferensi hasil ekspor |

Keduanya diterima. Bila yang dimuat berbentuk checkpoint pelatihan, jalur
`log_mel -> ctc_softmax` disarikan menjadi model inferensi — bobotnya sama persis,
yang ditinggalkan hanya cabang perhitungan loss yang tidak dipakai saat evaluasi.

In [ ]:
class _AugmentasiNonaktif(keras.layers.Layer):
    """Lapisan augmentasi apa pun, dimatikan untuk evaluasi.

    Menerima argumen konfigurasi apa pun, sehingga checkpoint yang memakai
    augmentasi berbeda (SpecAugment, VTLPAugment, dan yang menyusul) tetap dapat
    dimuat tanpa perlu menyalin implementasinya ke sini. Yang penting untuk
    pengukuran justru sebaliknya: augmentasi TIDAK boleh bekerja, sebab tiap
    pemanggilan akan mengubah masukan dan membuat CER berubah-ubah antar
    pengukuran.
    """

    def __init__(self, **kwargs):
        bawaan = {k: kwargs.pop(k) for k in ('name', 'dtype', 'trainable') if k in kwargs}
        self._ekstra = kwargs
        super().__init__(**bawaan)

    def call(self, inputs, training=None):
        return inputs

    def get_config(self):
        cfg = super().get_config()
        cfg.update(self._ekstra)
        return cfg


NAMA_AUGMENTASI = ['SpecAugment', 'VTLPAugment']
for _nama in NAMA_AUGMENTASI:
    keras.saving.register_keras_serializable(package='finsense', name=_nama)(
        type(_nama, (_AugmentasiNonaktif,), {}))


@keras.saving.register_keras_serializable(package='finsense', name='CTCLossLayer')
class CTCLossLayer(keras.layers.Layer):
    """Penghitung CTC loss saat pelatihan; saat evaluasi cukup meneruskan y_pred."""

    def __init__(self, name='ctc_loss', **kwargs):
        super().__init__(name=name, **kwargs)

    def call(self, y_true, y_pred, input_len, label_len):
        return y_pred

    def get_config(self):
        return super().get_config()


def muat_checkpoint(path):
    """Muat checkpoint -> (model inferensi, keterangan).

    Checkpoint pelatihan punya empat masukan dan cabang CTC loss. Untuk
    evaluasi hanya dibutuhkan jalur log_mel -> ctc_softmax, dan menyarikannya
    tidak mengubah bobot apa pun.
    """
    model = keras.models.load_model(path, compile=False)
    augmentasi = [lapis.__class__.__name__ for lapis in model.layers
                  if isinstance(lapis, _AugmentasiNonaktif)]
    if len(model.inputs) == 1:
        return model, {'bentuk': 'model inferensi', 'augmentasi': augmentasi}
    masukan = next((t for t in model.inputs if 'log_mel' in t.name), model.inputs[0])
    inferensi = keras.Model(masukan, model.get_layer('ctc_softmax').output,
                            name='inferensi_dari_checkpoint')
    return inferensi, {'bentuk': f'checkpoint pelatihan ({len(model.inputs)} masukan) '
                                 '-> disarikan jadi model inferensi',
                       'augmentasi': augmentasi}


with open(VOCAB_PATH, encoding='utf-8') as f:
    V = json.load(f)
idx2char = {int(k): v for k, v in V['idx2char'].items()}
BLANK = V['blank_idx']
CFG = {k: V[k] for k in ('sample_rate', 'n_mels', 'n_fft', 'win_length',
                         'hop_length', 'max_frames')}
print('kelas karakter:', len(idx2char), '+ 1 blank (indeks', BLANK, ')')
print('konfigurasi audio:', CFG)

MODEL, KETERANGAN = {}, {}
for skema, path in MODEL_SKEMA.items():
    if not path:
        print(f'\n[LEWATI] {skema}: checkpoint tidak ditemukan.')
        continue
    t0 = time.time()
    MODEL[skema], KETERANGAN[skema] = muat_checkpoint(path)
    print(f'\n[MUAT] {skema}')
    print(f'  berkas     : {path}')
    print(f'  bentuk     : {KETERANGAN[skema]["bentuk"]}')
    print(f'  augmentasi : {KETERANGAN[skema]["augmentasi"] or "tidak ada"} (dimatikan saat evaluasi)')
    print(f'  params     : {MODEL[skema].count_params():,} | waktu muat {time.time()-t0:.1f}s')

if not MODEL:
    raise FileNotFoundError(
        'Tidak ada checkpoint yang bisa dimuat.\n'
        f'Berkas .keras yang terlihat: {SEMUA_KERAS or "tidak ada"}\n'
        'Perbaiki dengan salah satu cara: (a) beri nama berkas dengan penanda '
        'skema, mis. finsense_ctc_A.keras / finsense_ctc_B.keras, atau '
        '(b) isi POLA_MODEL di sel konfigurasi dengan jalur berkasnya langsung.')

# Peringatan kesahihan 1: checkpoint dari korpus lain (mis. v12 yang dilatih di
# atas korpus Tahap 1) tidak sepadan dengan pembagian data pada notebook ini.
POLA_KORPUS_LAIN = ('v12', 'tahap1', 'tahap2', 'finetune', 'fine_tune')
for skema in MODEL:
    nama_berkas = os.path.basename(MODEL_SKEMA[skema]).lower()
    if any(pola in nama_berkas for pola in POLA_KORPUS_LAIN):
        print('\n' + '!' * 72)
        print(f'PERINGATAN: {skema} memakai {os.path.basename(MODEL_SKEMA[skema])}.')
        print('Nama berkas ini menandakan model yang dilatih di atas korpus Tahap 1,')
        print('yang diturunkan dari SELURUH rekaman manusia — termasuk rekaman yang di')
        print('notebook ini diperlakukan sebagai data uji. CER/WER-nya akan terlihat')
        print('bagus karena bocor, bukan karena modelnya lebih baik. Pakai checkpoint')
        print('yang dilatih langsung di atas finsense_expanded (v11 Skema A/B).')
        print('!' * 72)

# Peringatan kesahihan 2: satu checkpoint tidak boleh mewakili dua skenario.
if len(MODEL) == 2 and os.path.realpath(MODEL_SKEMA['speaker_dependent']) == \
        os.path.realpath(MODEL_SKEMA['speaker_independent']):
    print('\n' + '!' * 72)
    print('PERINGATAN: kedua skenario memakai checkpoint yang sama.')
    print('Bila checkpoint itu dilatih dengan Skema A, ia sudah pernah mendengar')
    print('SELURUH penutur, sehingga angka "speaker-independent" di bawah TIDAK sah')
    print('dan tidak boleh dikutip di laporan. Latih Skema B lebih dulu.')
    print('!' * 72)

# Peringatan kesahihan 3: kedua checkpoint harus berbeda HANYA pada pembagian data.
if len(MODEL) == 2:
    aug_sd = set(KETERANGAN['speaker_dependent']['augmentasi'])
    aug_si = set(KETERANGAN['speaker_independent']['augmentasi'])
    if aug_sd != aug_si:
        print('\n' + '!' * 72)
        print('PERINGATAN: kedua checkpoint dilatih dengan augmentasi yang BERBEDA.')
        print(f'  speaker_dependent   : {sorted(aug_sd) or "tidak ada"}')
        print(f'  speaker_independent : {sorted(aug_si) or "tidak ada"}')
        print('Selisih CER/WER antar skenario karena itu bercampur dua sebab:')
        print('pembagian data DAN perbedaan augmentasi. Untuk perbandingan yang')
        print('bersih, latih ulang salah satunya dengan augmentasi yang sama.')
        print('Bila tetap dilaporkan apa adanya, sebutkan perbedaan ini di laporan.')
        print('!' * 72)

**Insight:**

Seluruh lapisan augmentasi dibuat pass-through di sini. Kalau augmentasi ikut
aktif saat evaluasi, tiap pemanggilan akan mengubah masukan secara acak,
sehingga CER berbeda tiap kali diukur dan nilainya lebih buruk daripada
perilaku model yang sesungguhnya dilayankan.

Tiga peringatan di akhir sel menjaga hal-hal yang paling mudah terlewat karena
notebook tetap berjalan mulus meski terjadi: memakai satu checkpoint untuk dua
skenario, memakai model dari korpus lain, dan — yang paling halus — membandingkan
dua checkpoint yang berbeda bukan hanya pada pembagian datanya. Perhatikan baris
`augmentasi` pada keluaran di atas: bila kedua skenario memakai daftar yang
berbeda, selisih CER yang terbaca bukan murni pengaruh penutur baru.

# **METRIK: CER, WER, DAN RINCIAN KESALAHAN**

**CER (*Character Error Rate*)** dan **WER (*Word Error Rate*)** dihitung dengan
rumus yang sama, berbeda pada satuan yang dibandingkan:

$$\mathrm{CER} = \frac{S + D + I}{N}$$

dengan $S$ substitusi, $D$ penghapusan (*deletion*), $I$ penyisipan
(*insertion*), dan $N$ jumlah karakter pada teks acuan. WER memakai satuan kata.
Ketiga jenis kesalahan diperoleh dari penelusuran balik matriks jarak
Levenshtein, sehingga bukan hanya total kesalahan yang diketahui, melainkan juga
jenisnya — informasi yang menentukan arah perbaikan.

Dua cara agregasi dilaporkan, dan keduanya menjawab pertanyaan berbeda:

- **CER korpus** — seluruh kesalahan dibagi seluruh karakter acuan. Ini angka
  baku pada literatur ASR dan yang dikutip sebagai hasil utama.
- **CER rerata per ucapan** — CER dihitung per ucapan lalu dirata-rata. Ucapan
  pendek mendapat bobot yang sama dengan ucapan panjang, dan sebarannya dapat
  digambar sebagai boxplot.

In [ ]:
def jarak_edit(ref, hyp):
    """Levenshtein dengan rincian S/D/I lewat penelusuran balik.

    ref dan hyp berupa urutan apa pun yang dapat dibandingkan: string (untuk
    CER) atau daftar kata (untuk WER). Mengembalikan dict berisi sub, hap, sis,
    total, dan n (panjang acuan).
    """
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), dtype=np.int32)
    d[:, 0] = np.arange(n + 1)
    d[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            biaya = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1,        # penghapusan
                          d[i, j - 1] + 1,        # penyisipan
                          d[i - 1, j - 1] + biaya)  # substitusi/cocok

    sub = hap = sis = 0
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1] and d[i, j] == d[i - 1, j - 1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + 1:
            sub += 1; i, j = i - 1, j - 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            hap += 1; i -= 1
        else:
            sis += 1; j -= 1
    return {'sub': sub, 'hap': hap, 'sis': sis, 'total': int(d[n, m]), 'n': n}


def bootstrap_ci(kesalahan, satuan, n_ulang=N_BOOTSTRAP, alpha=0.05, seed=SEED):
    """Selang kepercayaan bootstrap untuk rasio korpus sum(kesalahan)/sum(satuan).

    Ucapan yang di-resample, bukan karakternya — karena ucapanlah unit
    pengamatan yang saling bebas di sini.
    """
    kesalahan = np.asarray(kesalahan, dtype=np.float64)
    satuan = np.asarray(satuan, dtype=np.float64)
    rng = np.random.RandomState(seed)
    n = len(kesalahan)
    if n == 0:
        return float('nan'), float('nan')
    idx = rng.randint(0, n, size=(n_ulang, n))
    rasio = kesalahan[idx].sum(axis=1) / np.maximum(satuan[idx].sum(axis=1), 1e-9)
    return float(np.quantile(rasio, alpha / 2)), float(np.quantile(rasio, 1 - alpha / 2))


# Pemeriksaan cepat agar rumusnya tidak dipercaya begitu saja.
uji = jarak_edit('jual aqua dua', 'jual akua dua')
assert uji['sub'] == 1 and uji['hap'] == 0 and uji['sis'] == 0, uji
uji2 = jarak_edit(list('abc'), list('ac'))
assert uji2['hap'] == 1 and uji2['total'] == 1, uji2
uji3 = jarak_edit(list('ac'), list('abc'))
assert uji3['sis'] == 1 and uji3['total'] == 1, uji3
print('Fungsi jarak edit lolos uji: substitusi, penghapusan, dan penyisipan terhitung benar.')

**Insight:**

Fungsi jarak edit diuji lebih dulu pada tiga kasus yang jawabannya sudah
diketahui. Kekeliruan pada penelusuran balik tidak akan membuat notebook gagal —
ia hanya menghasilkan angka yang salah dan tampak wajar, dan kekeliruan semacam
itu biasanya baru ketahuan setelah tabelnya terlanjur masuk laporan.

# **EKSTRAKSI FITUR & DECODING**

Tahap ini menyiapkan dua fungsi yang menghubungkan berkas audio dengan keluaran
model: ekstraksi *log-mel spectrogram* dan *greedy CTC decoding*.

Parameter ekstraksi fitur diambil dari `char_vocab.json` milik model, bukan
ditulis ulang. Selisih sekecil apa pun antara fitur saat pelatihan, saat
evaluasi, dan saat pelayanan akan membuat angka pada notebook ini tidak
menggambarkan perilaku model di aplikasi.

In [ ]:
def ekstrak_logmel(path):
    """Berkas audio -> log-mel ternormalisasi, di-pad/potong ke max_frames."""
    y, _ = librosa.load(path, sr=CFG['sample_rate'])
    S = librosa.feature.melspectrogram(
        y=y, sr=CFG['sample_rate'], n_mels=CFG['n_mels'], n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'], win_length=CFG['win_length'])
    S_db = librosa.power_to_db(S, ref=np.max).T
    S_db = ((S_db - S_db.mean()) / (S_db.std() + 1e-8)).astype(np.float32)
    T = S_db.shape[0]
    if T < CFG['max_frames']:
        return np.vstack([S_db, np.zeros((CFG['max_frames'] - T, CFG['n_mels']), np.float32)])
    return S_db[:CFG['max_frames']]


def greedy_decode(probs_batch):
    """Argmax per frame -> gabungkan pengulangan -> buang blank.

    probs_batch: (batch, waktu, kelas). Mengembalikan daftar string.
    Ini decoder yang sama dengan yang dipakai layanan REST (ai-service), supaya
    angka pada notebook ini sebanding dengan perilaku aplikasi.
    """
    hasil = []
    for ids in np.argmax(probs_batch, axis=-1):
        keluar, sebelum = [], -1
        for i in ids:
            i = int(i)
            if i != BLANK and i != sebelum:
                keluar.append(idx2char.get(i, ''))
            sebelum = i
        hasil.append(''.join(keluar))
    return hasil


contoh = ekstrak_logmel(os.path.join(AUDIO_DIR, df['filename'].iloc[0]))
print('bentuk fitur satu ucapan:', contoh.shape, '| dtype:', contoh.dtype)
print('rerata:', round(float(contoh.mean()), 3), '| simpangan baku:', round(float(contoh.std()), 3))

**Insight:**

Fitur dinormalisasi per ucapan (*cepstral mean and variance normalization*)
sebelum di-*pad*, bukan sesudahnya. Urutan ini penting: bila padding nol ikut
dihitung dalam rerata dan simpangan baku, ucapan pendek akan ternormalisasi
berbeda dari ucapan panjang, dan model menerima masukan yang tidak sepadan
dengan data latihnya.

# **EVALUASI**

Tahap ini menjalankan model pada data uji tiap skenario, lalu menghitung
kesalahan tingkat karakter dan tingkat kata untuk setiap ucapan.

Hasil per ucapan disimpan ke `transkripsi_<skenario>.csv`. Berkas itu bukan
sekadar arsip: notebook `08_evaluasi_nlu.ipynb` memakainya sebagai masukan,
sehingga evaluasi NLU berjalan di atas teks hasil transkripsi yang sama persis,
bukan hasil pengukuran ulang yang bisa berbeda.

In [ ]:
def evaluasi_skema(skema, model, indeks):
    """Transkripsikan data uji satu skenario, kembalikan DataFrame per ucapan."""
    if MAKS_SAMPEL:
        rng = np.random.RandomState(SEED)
        indeks = sorted(rng.choice(indeks, size=min(MAKS_SAMPEL, len(indeks)),
                                   replace=False).tolist())

    print(f'\n=== {skema} — {len(indeks):,} ucapan ===')
    baris, t_mulai = [], time.time()

    for awal in range(0, len(indeks), BATCH_INFER):
        potong = indeks[awal:awal + BATCH_INFER]
        X = np.stack([ekstrak_logmel(os.path.join(AUDIO_DIR, df['filename'].iloc[i]))
                      for i in potong])
        probs = model.predict(X, verbose=0)
        for i, hyp in zip(potong, greedy_decode(probs)):
            acuan = df['teks'].iloc[i]
            k = jarak_edit(acuan, hyp)
            w = jarak_edit(acuan.split(), hyp.split())
            baris.append({
                'filename': df['filename'].iloc[i],
                'skema': skema,
                'penutur': int(df['penutur'].iloc[i]),
                'sintetis': bool(df['penutur'].iloc[i] == -1),
                'bocor_varian': bool(df.get('bocor_' + skema, pd.Series(False, index=df.index)).iloc[i]),
                'teks_acuan': acuan,
                'teks_prediksi': hyp,
                'n_karakter': k['n'], 'sub_kar': k['sub'], 'hap_kar': k['hap'],
                'sis_kar': k['sis'], 'err_kar': k['total'],
                'n_kata': w['n'], 'sub_kata': w['sub'], 'hap_kata': w['hap'],
                'sis_kata': w['sis'], 'err_kata': w['total'],
                'cer_ucapan': k['total'] / max(k['n'], 1),
                'wer_ucapan': w['total'] / max(w['n'], 1),
                'tepat': acuan == hyp,
            })
        selesai = min(awal + BATCH_INFER, len(indeks))
        if selesai % (BATCH_INFER * 10) == 0 or selesai == len(indeks):
            lewat = time.time() - t_mulai
            print(f'  {selesai:,}/{len(indeks):,} ucapan | {lewat:.0f}s '
                  f'({1000*lewat/selesai:.0f} ms/ucapan)')

    return pd.DataFrame(baris)


HASIL = {}
for skema, model in MODEL.items():
    _, _, idx_uji = SPLIT[skema]
    HASIL[skema] = evaluasi_skema(skema, model, idx_uji)
    keluaran_csv = os.path.join(KELUARAN, f'transkripsi_{skema}.csv')
    HASIL[skema].to_csv(keluaran_csv, index=False)
    print(f'  tersimpan: {keluaran_csv}')

per_ucapan = pd.concat(HASIL.values(), ignore_index=True)
print(f'\nTotal ucapan dievaluasi: {len(per_ucapan):,}')
per_ucapan.head()

**Insight:**

Kolom `bocor_varian` ikut disimpan per ucapan, bukan hanya dirangkum. Dengan
begitu setiap angka agregat pada bagian berikutnya dapat dihitung ulang tanpa
ucapan yang bocor, dan klaim "kebocoran ini kecil pengaruhnya" bisa diperiksa,
bukan sekadar dinyatakan.

# **HASIL: TABEL CER & WER**

Tahap ini merangkum hasil per ucapan menjadi tabel utama untuk laporan. Selain
CER dan WER korpus, ditampilkan pula selang kepercayaan 95% hasil *bootstrap*,
rerata per ucapan, *exact match* (persentase ucapan yang transkripsinya persis
sama dengan acuan), serta CER pada subset tanpa kebocoran varian augmentasi.

In [ ]:
def rangkum(dfh, label):
    cer = dfh.err_kar.sum() / max(dfh.n_karakter.sum(), 1)
    wer = dfh.err_kata.sum() / max(dfh.n_kata.sum(), 1)
    cer_lo, cer_hi = bootstrap_ci(dfh.err_kar.values, dfh.n_karakter.values)
    wer_lo, wer_hi = bootstrap_ci(dfh.err_kata.values, dfh.n_kata.values)
    bersih = dfh[~dfh.bocor_varian]
    cer_bersih = (bersih.err_kar.sum() / max(bersih.n_karakter.sum(), 1)
                  if len(bersih) else float('nan'))
    return {
        'skenario': label,
        'n_ucapan': len(dfh),
        'n_penutur_uji': int(dfh.loc[dfh.penutur >= 0, 'penutur'].nunique()),
        'CER_%': round(100 * cer, 2),
        'CER_CI95': f'[{100*cer_lo:.2f}; {100*cer_hi:.2f}]',
        'WER_%': round(100 * wer, 2),
        'WER_CI95': f'[{100*wer_lo:.2f}; {100*wer_hi:.2f}]',
        'CER_rerata_ucapan_%': round(100 * dfh.cer_ucapan.mean(), 2),
        'exact_match_%': round(100 * dfh.tepat.mean(), 2),
        'substitusi_%': round(100 * dfh.sub_kar.sum() / max(dfh.err_kar.sum(), 1), 1),
        'penghapusan_%': round(100 * dfh.hap_kar.sum() / max(dfh.err_kar.sum(), 1), 1),
        'penyisipan_%': round(100 * dfh.sis_kar.sum() / max(dfh.err_kar.sum(), 1), 1),
        'CER_tanpa_kebocoran_%': round(100 * cer_bersih, 2),
    }


LABEL = {'speaker_dependent': 'Speaker-dependent', 'speaker_independent': 'Speaker-independent'}
tabel_utama = pd.DataFrame([rangkum(HASIL[s], LABEL[s]) for s in HASIL])
tabel_utama.to_csv(os.path.join(KELUARAN, 'tabel_cer_wer.csv'), index=False)

print(tabel_utama.to_string(index=False))
print()
print('Tersimpan:', os.path.join(KELUARAN, 'tabel_cer_wer.csv'))
tabel_utama

**Insight:**

Tiga kolom pantas dibaca berdampingan.

`CER_%` dan `CER_CI95` — selang kepercayaan menunjukkan seberapa jauh angka itu
dapat bergeser bila pengujian diulang dengan sampel ucapan lain. Dua skenario
yang selangnya bertumpang tindih tidak boleh dinyatakan berbeda hanya karena
angka tengahnya berbeda.

`substitusi_% / penghapusan_% / penyisipan_%` — komposisi ini mengarahkan
perbaikan. Substitusi yang dominan berarti model mendengar bunyi tetapi salah
memilih huruf (masalah akustik/leksikon). Penghapusan yang dominan berarti model
melewatkan bagian ucapan — biasanya ucapan terlalu cepat, terlalu pelan, atau
terpotong jendela masukan.

`exact_match_%` adalah ukuran paling keras: satu huruf meleset membuat seluruh
ucapan dihitung salah. Nilainya selalu jauh lebih rendah daripada 100 − CER, dan
memang begitu seharusnya.

# **PERBANDINGAN VISUAL**

Tahap ini menyajikan hasil dalam tiga tampilan: perbandingan CER/WER antar
skenario beserta selang kepercayaannya, sebaran CER per ucapan, dan komposisi
jenis kesalahan.

In [ ]:
skenario = list(HASIL.keys())
warna = {'speaker_dependent': '#2563eb', 'speaker_independent': '#dc2626'}

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) CER & WER dengan selang kepercayaan
lebar, posisi = 0.35, np.arange(2)
for k, s in enumerate(skenario):
    d = HASIL[s]
    cer = d.err_kar.sum() / max(d.n_karakter.sum(), 1)
    wer = d.err_kata.sum() / max(d.n_kata.sum(), 1)
    cl, ch = bootstrap_ci(d.err_kar.values, d.n_karakter.values)
    wl, wh = bootstrap_ci(d.err_kata.values, d.n_kata.values)
    nilai = np.array([cer, wer]) * 100
    bawah = nilai - np.array([cl, wl]) * 100
    atas = np.array([ch, wh]) * 100 - nilai
    bar = ax[0].bar(posisi + k * lebar, nilai, lebar, label=LABEL[s],
                    color=warna[s], yerr=[bawah, atas], capsize=4)
    ax[0].bar_label(bar, fmt='%.2f', padding=6, fontsize=9)
ax[0].set_xticks(posisi + lebar / 2); ax[0].set_xticklabels(['CER', 'WER'])
ax[0].set_ylabel('tingkat kesalahan (%)')
ax[0].set_title('(a) CER & WER per skenario\n(galat = selang kepercayaan 95%)')
ax[0].legend(fontsize=8); ax[0].grid(axis='y', alpha=0.3)

# (b) sebaran CER per ucapan
ax[1].boxplot([HASIL[s].cer_ucapan.values * 100 for s in skenario], showfliers=False)
# set_xticklabels dipakai, bukan argumen labels=, karena nama argumen itu
# berganti (labels -> tick_labels) pada matplotlib 3.9 dan akan error di salah
# satu versi.
ax[1].set_xticks(range(1, len(skenario) + 1))
ax[1].set_xticklabels([LABEL[s].replace('-', '-\n') for s in skenario])
ax[1].set_ylabel('CER per ucapan (%)')
ax[1].set_title('(b) Sebaran CER per ucapan\n(pencilan disembunyikan)')
ax[1].grid(axis='y', alpha=0.3)

# (c) komposisi jenis kesalahan
jenis = ['sub_kar', 'hap_kar', 'sis_kar']
nama_jenis = ['substitusi', 'penghapusan', 'penyisipan']
dasar = np.zeros(len(skenario))
for j, nama in zip(jenis, nama_jenis):
    nilai = np.array([100 * HASIL[s][j].sum() / max(HASIL[s].err_kar.sum(), 1)
                      for s in skenario])
    ax[2].bar([LABEL[s].replace('-', '-\n') for s in skenario], nilai, bottom=dasar, label=nama)
    dasar += nilai
ax[2].set_ylabel('porsi dari total kesalahan (%)')
ax[2].set_title('(c) Komposisi jenis kesalahan karakter')
ax[2].legend(fontsize=8); ax[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(KELUARAN, 'perbandingan_cer_wer.png'), dpi=150)
plt.show()

**Insight:**

Panel (b) menjelaskan hal yang tidak terlihat pada panel (a). CER korpus adalah
satu angka; sebaran per ucapan menunjukkan apakah angka itu mewakili kebanyakan
ucapan atau justru hasil tarik-menarik antara sebagian besar ucapan yang nyaris
sempurna dan segelintir ucapan yang gagal total. Median yang jauh di bawah CER
korpus adalah tanda pola yang kedua, dan artinya perbaikan sebaiknya diarahkan
ke ucapan sulitnya, bukan ke model secara keseluruhan.

# **ANALISIS PER PENUTUR**

Tahap ini membedah hasil menurut penutur. Pada skenario *speaker-independent*
tiap penutur uji adalah satu orang yang sama sekali baru bagi model, sehingga
selisih antar-penutur memperlihatkan seberapa besar kinerja bergantung pada
karakteristik suara — logat, kecepatan bicara, dan kondisi perekaman.

In [ ]:
baris = []
for s in skenario:
    d = HASIL[s]
    for pen, grup in d[d.penutur >= 0].groupby('penutur'):
        baris.append({
            'skenario': LABEL[s],
            'penutur': int(pen),
            'n_ucapan': len(grup),
            'CER_%': round(100 * grup.err_kar.sum() / max(grup.n_karakter.sum(), 1), 2),
            'WER_%': round(100 * grup.err_kata.sum() / max(grup.n_kata.sum(), 1), 2),
            'exact_match_%': round(100 * grup.tepat.mean(), 1),
        })
tabel_penutur = pd.DataFrame(baris).sort_values(['skenario', 'CER_%'])
tabel_penutur.to_csv(os.path.join(KELUARAN, 'cer_per_penutur.csv'), index=False)

fig, ax = plt.subplots(1, len(skenario), figsize=(6.5 * len(skenario), 4), squeeze=False)
for k, s in enumerate(skenario):
    sub = tabel_penutur[tabel_penutur.skenario == LABEL[s]]
    a = ax[0][k]
    bar = a.bar(sub.penutur.astype(str), sub['CER_%'], color=warna[s])
    a.bar_label(bar, fmt='%.1f', fontsize=8, padding=2)
    rerata = 100 * HASIL[s].err_kar.sum() / max(HASIL[s].n_karakter.sum(), 1)
    a.axhline(rerata, color='black', ls='--', lw=1,
              label=f'CER korpus {rerata:.2f}%')
    a.set_title(f'CER per penutur — {LABEL[s]}')
    a.set_xlabel('id penutur'); a.set_ylabel('CER (%)')
    a.legend(fontsize=8); a.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(KELUARAN, 'cer_per_penutur.png'), dpi=150)
plt.show()

for s in skenario:
    sub = tabel_penutur[tabel_penutur.skenario == LABEL[s]]
    if len(sub) >= 2:
        print(f'{LABEL[s]}: CER terbaik {sub["CER_%"].min():.2f}% (penutur '
              f'{int(sub.loc[sub["CER_%"].idxmin(), "penutur"])}), '
              f'terburuk {sub["CER_%"].max():.2f}% (penutur '
              f'{int(sub.loc[sub["CER_%"].idxmax(), "penutur"])}), '
              f'rentang {sub["CER_%"].max() - sub["CER_%"].min():.2f} poin')
tabel_penutur.head(12)

**Insight:**

Rentang antar-penutur adalah informasi yang hilang bila hanya satu angka CER
yang dilaporkan. Rentang yang lebar berarti pengalaman pengguna tidak seragam:
sebagian pedagang akan merasa sistemnya akurat, sebagian lain merasa sering
salah, padahal modelnya sama.

Pada skenario *speaker-independent* jumlah penutur ujinya sedikit (tiga orang),
jadi angka per penutur di sini menggambarkan orang-orang tersebut, bukan
populasi pedagang secara umum. Untuk klaim yang lebih luas diperlukan penutur
uji yang lebih banyak — itu batasan data, bukan batasan metode.

# **UJI SIGNIFIKANSI SELISIH ANTAR-SKENARIO**

Tahap ini menguji apakah selisih CER antara kedua skenario cukup besar untuk
disebut nyata, bukan kebetulan akibat ucapan mana yang kebetulan masuk data uji.

Dua uji dipakai karena keduanya menjawab hal berbeda:

- **Bootstrap selisih CER korpus** — menaksir selang kepercayaan selisih. Bila
  selang memuat nol, selisihnya tidak dapat dinyatakan ada.
- **Uji Mann-Whitney U** — uji non-parametrik atas sebaran CER per ucapan.
  Dipilih karena sebaran CER sangat menceng (banyak ucapan bernilai nol),
  sehingga uji-t yang mengandaikan kenormalan tidak tepat.

In [ ]:
if len(HASIL) < 2:
    print('Hanya satu skenario yang terevaluasi — uji perbandingan dilewati.')
else:
    a, b = 'speaker_dependent', 'speaker_independent'
    da, db = HASIL[a], HASIL[b]

    cer_a = da.err_kar.sum() / max(da.n_karakter.sum(), 1)
    cer_b = db.err_kar.sum() / max(db.n_karakter.sum(), 1)

    rng = np.random.RandomState(SEED)
    selisih = []
    for _ in range(N_BOOTSTRAP):
        ia = rng.randint(0, len(da), len(da))
        ib = rng.randint(0, len(db), len(db))
        ca = da.err_kar.values[ia].sum() / max(da.n_karakter.values[ia].sum(), 1)
        cb = db.err_kar.values[ib].sum() / max(db.n_karakter.values[ib].sum(), 1)
        selisih.append(cb - ca)
    selisih = np.array(selisih)
    lo, hi = np.quantile(selisih, [0.025, 0.975])

    print(f'CER {LABEL[a]:22s}: {100*cer_a:.2f}%')
    print(f'CER {LABEL[b]:22s}: {100*cer_b:.2f}%')
    print(f'Selisih (SI - SD)          : {100*(cer_b-cer_a):+.2f} poin persen')
    print(f'Selang kepercayaan 95%     : [{100*lo:+.2f}; {100*hi:+.2f}] poin persen')
    print('Kesimpulan bootstrap       :',
          'selisih NYATA (selang tidak memuat nol)' if lo > 0 or hi < 0
          else 'selisih TIDAK dapat dinyatakan (selang memuat nol)')

    try:
        from scipy.stats import mannwhitneyu
        u, p = mannwhitneyu(da.cer_ucapan.values, db.cer_ucapan.values,
                            alternative='two-sided')
        n1, n2 = len(da), len(db)
        # Ukuran efek rank-biserial: 0 = tidak ada perbedaan, 1 = terpisah penuh.
        rb = 1 - (2 * u) / (n1 * n2)
        print(f'\nMann-Whitney U             : U={u:,.0f} | p={p:.3e}')
        print(f'Ukuran efek (rank-biserial): {rb:+.3f}')
        print('Kesimpulan uji             :',
              'berbeda nyata (p < 0.05)' if p < 0.05 else 'tidak berbeda nyata (p >= 0.05)')
    except ImportError:
        print('\nscipy tidak tersedia — uji Mann-Whitney dilewati, '
              'kesimpulan bootstrap di atas tetap berlaku.')

**Insight:**

Nilai p menjawab "apakah ada selisih", bukan "apakah selisihnya penting".
Dengan ratusan ucapan, selisih sekecil apa pun bisa keluar sebagai signifikan.
Karena itu ukuran efek dan selang kepercayaan dilaporkan bersamaan: keputusan
rekayasa — misalnya apakah model perlu dilatih ulang dengan lebih banyak
penutur — ditentukan oleh besar selisihnya dalam poin persen, bukan oleh nilai
p.

# **UJI SILANG: KEDUA MODEL PADA HIMPUNAN UJI YANG SAMA**

Perbandingan di bagian sebelumnya punya satu kelemahan yang tidak bisa
dihilangkan dengan statistik: **kedua skenario diuji pada ucapan yang berbeda**.
Skenario *speaker-dependent* diuji pada 765 ucapan campuran manusia dan
sintetis, sedangkan *speaker-independent* pada 180 ucapan manusia dari tiga
penutur. Selisih yang terbaca karena itu bercampur dengan perbedaan tingkat
kesulitan kedua himpunan uji.

Bagian ini menghilangkan percampuran itu dengan cara paling langsung:
**kedua model dijalankan pada himpunan ucapan yang persis sama**, yaitu data uji
Skema B — 180 ucapan manusia dari penutur 0, 15, dan 17.

Bedanya hanya satu hal, dan itulah yang sedang diukur:

| model | tiga penutur itu saat pelatihan |
|---|---|
| checkpoint Skema A | **pernah didengar** (ada di data latihnya) |
| checkpoint Skema B | **tidak pernah didengar** (dikunci di luar) |

Karena ucapannya sama persis, selisihnya dapat diuji berpasangan dengan uji
Wilcoxon *signed-rank* — uji yang tepat untuk pengamatan berpasangan dan tidak
mengandaikan sebaran normal.

In [ ]:
if len(MODEL) < 2:
    print('[LEWATI] Uji silang membutuhkan kedua checkpoint.')
    hasil_silang = None
else:
    idx_uji_bersama = SPLIT['speaker_independent'][2]
    penutur_bersama = sorted(set(df.loc[idx_uji_bersama, 'penutur']) - {-1})
    print(f'Himpunan uji bersama: {len(idx_uji_bersama)} ucapan dari penutur {penutur_bersama}')
    print('(Seluruhnya rekaman manusia; bagi checkpoint Skema A ketiga penutur ini '
          'ADA di data latih, bagi Skema B tidak.)')

    hasil_silang = {}
    for skema, model in MODEL.items():
        hasil_silang[skema] = evaluasi_skema(f'{skema}_pada_uji_bersama',
                                             model, idx_uji_bersama)

    baris = []
    for skema, d in hasil_silang.items():
        baris.append({
            'checkpoint': LABEL[skema],
            'penutur uji bagi model ini': 'pernah dilatih' if skema == 'speaker_dependent'
                                          else 'belum pernah',
            'n_ucapan': len(d),
            'CER_%': round(100 * d.err_kar.sum() / max(d.n_karakter.sum(), 1), 2),
            'WER_%': round(100 * d.err_kata.sum() / max(d.n_kata.sum(), 1), 2),
            'exact_match_%': round(100 * d.tepat.mean(), 2),
        })
    tabel_silang = pd.DataFrame(baris)
    tabel_silang.to_csv(os.path.join(KELUARAN, 'uji_silang_himpunan_sama.csv'), index=False)
    print()
    print(tabel_silang.to_string(index=False))

    # Simpan hasil per ucapan kedua model: tanpa ini, uji berpasangan di bawah
    # tidak dapat dihitung ulang dari berkas keluaran.
    pd.concat(hasil_silang.values(), ignore_index=True).to_csv(
        os.path.join(KELUARAN, 'transkripsi_uji_silang.csv'), index=False)

    # Rincian per penutur: rerata dapat menyembunyikan arah yang berlawanan
    # antar penutur, dan dengan tiga penutur hal itu justru lazim terjadi.
    print('\nCER per penutur pada himpunan uji yang sama:')
    baris_penutur = []
    for pen in penutur_bersama:
        sel = {}
        for skema, d in hasil_silang.items():
            g = d[d.penutur == pen]
            sel[skema] = 100 * g.err_kar.sum() / max(g.n_karakter.sum(), 1)
        baris_penutur.append({
            'penutur': pen,
            'CER_model_A_%': round(sel['speaker_dependent'], 2),
            'CER_model_B_%': round(sel['speaker_independent'], 2),
            'selisih_poin': round(sel['speaker_independent'] - sel['speaker_dependent'], 2),
        })
    tabel_penutur_silang = pd.DataFrame(baris_penutur)
    tabel_penutur_silang.to_csv(os.path.join(KELUARAN, 'uji_silang_per_penutur.csv'), index=False)
    print(tabel_penutur_silang.to_string(index=False))

    # ── Uji berpasangan: ucapan yang sama dinilai dua model ──
    a = hasil_silang['speaker_dependent'].set_index('filename')
    b = hasil_silang['speaker_independent'].set_index('filename')
    bersama = [f for f in a.index if f in b.index]
    cer_a = a.loc[bersama, 'cer_ucapan'].values
    cer_b = b.loc[bersama, 'cer_ucapan'].values
    selisih = cer_b - cer_a

    print(f'\nSebaran CER per ucapan (menunjukkan BENTUK kesalahan, bukan hanya besarnya):')
    for nama, v in (('model A (penutur dikenal)', cer_a), ('model B (penutur baru)', cer_b)):
        print(f'  {nama:26s} median {100*np.median(v):.2f}% | '
              f'persentil-90 {100*np.quantile(v, 0.9):.2f}% | maks {100*v.max():.2f}% | '
              f'ucapan sempurna {(v == 0).sum()}')

    print(f'\nUcapan berpasangan: {len(bersama)}')
    print(f'  model Skema A (penutur dikenal)  : CER rerata per ucapan {100*cer_a.mean():.2f}%')
    print(f'  model Skema B (penutur baru)     : CER rerata per ucapan {100*cer_b.mean():.2f}%')
    print(f'  ucapan yang memburuk bagi Skema B: {(selisih > 0).sum()} '
          f'| membaik: {(selisih < 0).sum()} | sama: {(selisih == 0).sum()}')

    try:
        from scipy.stats import wilcoxon
        bukan_nol = selisih[selisih != 0]
        if len(bukan_nol):
            stat, p = wilcoxon(cer_a[selisih != 0], cer_b[selisih != 0])
            print(f'\nWilcoxon signed-rank: W={stat:,.0f} | p={p:.3e}')
            print('  Kesimpulan:', 'berbeda nyata (p < 0.05)' if p < 0.05
                  else 'tidak berbeda nyata (p >= 0.05)')
        else:
            print('\nSeluruh pasangan identik — tidak ada yang dapat diuji.')
    except ImportError:
        print('\nscipy tidak tersedia — uji Wilcoxon dilewati.')

    # Selang kepercayaan selisih CER korpus, di-resample berpasangan.
    rng = np.random.RandomState(SEED)
    err_a = a.loc[bersama, 'err_kar'].values; n_a = a.loc[bersama, 'n_karakter'].values
    err_b = b.loc[bersama, 'err_kar'].values; n_b = b.loc[bersama, 'n_karakter'].values
    beda = []
    for _ in range(N_BOOTSTRAP):
        i = rng.randint(0, len(bersama), len(bersama))
        beda.append(err_b[i].sum() / max(n_b[i].sum(), 1) - err_a[i].sum() / max(n_a[i].sum(), 1))
    lo, hi = np.quantile(beda, [0.025, 0.975])
    cer_korpus_a = err_a.sum() / max(n_a.sum(), 1)
    cer_korpus_b = err_b.sum() / max(n_b.sum(), 1)
    print(f'\nSelisih CER korpus (Skema B - Skema A) pada ucapan yang sama: '
          f'{100*(cer_korpus_b - cer_korpus_a):+.2f} poin persen')
    print(f'Selang kepercayaan 95% (resample berpasangan): [{100*lo:+.2f}; {100*hi:+.2f}]')
    print('Kesimpulan:', 'selisih NYATA' if lo > 0 or hi < 0 else 'selisih TIDAK dapat dinyatakan')

**Insight:**

Inilah angka yang paling pantas dipakai menjawab RM-2 — dengan satu syarat yang
harus diperiksa lebih dulu di tabel per penutur dan sebaran CER di atas.

Bila arah selisihnya **berlawanan antar penutur** — satu penutur jauh lebih baik
pada model A, penutur lain justru lebih baik pada model B — maka yang terukur
bukan pengaruh pengenalan penutur, melainkan perbedaan mutu kedua checkpoint
secara keseluruhan. Begitu pula bila model dengan CER lebih tinggi justru punya
lebih banyak ucapan sempurna: itu tanda kesalahannya menumpuk di sedikit ucapan
yang gagal total, bukan tersebar merata seperti yang diharapkan dari efek
penutur baru.

Dalam dua keadaan itu, kesimpulan yang sah adalah bahwa perbandingan ini belum
mampu memisahkan pengaruh penutur, dan penyebab yang paling mungkin adalah
kedua checkpoint tidak dilatih pada jumlah data yang sama. Pada tabel sebelumnya,
selisih antar skenario bercampur dengan perbedaan himpunan uji: data uji Skema A
memuat ucapan sintetis yang lebih mudah, dan data uji Skema B hanya tiga
penutur. Di sini ucapannya identik, penuturnya identik, rekamannya identik —
satu-satunya yang berbeda adalah apakah suara ketiga penutur itu pernah masuk
data latih model yang menilainya.

Dua hal yang tetap perlu disebutkan saat melaporkan angka ini. Pertama, kedua
checkpoint tidak hanya berbeda pada penutur yang dilihat, tetapi juga pada
jumlah data latihnya — Skema B memakai seluruh ucapan sintetis, sedangkan Skema
A menyisihkan sebagian untuk validasi dan uji. Kedua, ketiga penutur uji adalah
sampel kecil; selang kepercayaan di atas menggambarkan ketidakpastian atas
ucapan, bukan atas populasi penutur Indonesia.

Untuk memisahkan pengaruh penutur secara bersih, kedua model harus dilatih pada
jumlah data yang sama dan diuji pada ucapan yang sama. Caranya: sisihkan
sebagian ucapan dari ketiga penutur uji (misalnya 15 dari 60) sebagai data uji
bagi **kedua** model; model *speaker-independent* dilatih tanpa ketiga penutur
itu sama sekali, sedangkan model *speaker-dependent* dilatih pada data yang
sama persis ditambah 45 ucapan sisanya dari tiap penutur uji. Dengan begitu
satu-satunya perbedaan adalah keterpaparan pada suara penutur uji.

# **ANALISIS KESALAHAN**

Tahap ini memeriksa kesalahan yang sesungguhnya terjadi: ucapan dengan CER
tertinggi, dan pasangan kata yang paling sering tertukar. Tujuannya menemukan
pola yang dapat ditindaklanjuti — apakah kesalahan menumpuk pada nama produk,
pada kata bilangan, atau pada kata harga.

In [ ]:
def operasi_kata(ref, hyp):
    """Daftar operasi penyuntingan tingkat kata: (jenis, kata_acuan, kata_prediksi)."""
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), dtype=np.int32)
    d[:, 0] = np.arange(n + 1); d[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            biaya = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i, j] = min(d[i-1, j] + 1, d[i, j-1] + 1, d[i-1, j-1] + biaya)
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1] and d[i, j] == d[i-1, j-1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and d[i, j] == d[i-1, j-1] + 1:
            ops.append(('substitusi', ref[i-1], hyp[j-1])); i, j = i - 1, j - 1
        elif i > 0 and d[i, j] == d[i-1, j] + 1:
            ops.append(('penghapusan', ref[i-1], '')); i -= 1
        else:
            ops.append(('penyisipan', '', hyp[j-1])); j -= 1
    return ops[::-1]


for s in skenario:
    d = HASIL[s]
    print('=' * 78)
    print(f'{LABEL[s]} — 5 ucapan dengan CER tertinggi')
    print('=' * 78)
    for _, r in d.nlargest(5, 'cer_ucapan').iterrows():
        print(f'  [{r.filename}] CER {100*r.cer_ucapan:.1f}% | penutur {r.penutur}')
        print(f'    acuan   : {r.teks_acuan}')
        print(f'    prediksi: {r.teks_prediksi}')

    pasangan = collections.Counter()
    hilang = collections.Counter()
    for _, r in d.iterrows():
        for jenis, ka, kh in operasi_kata(r.teks_acuan.split(), r.teks_prediksi.split()):
            if jenis == 'substitusi':
                pasangan[(ka, kh)] += 1
            elif jenis == 'penghapusan':
                hilang[ka] += 1
    print(f'\n  10 substitusi kata terbanyak ({LABEL[s]}):')
    for (ka, kh), n in pasangan.most_common(10):
        print(f'    {n:4d}x  "{ka}" -> "{kh}"')
    print(f'\n  10 kata yang paling sering hilang ({LABEL[s]}):')
    for ka, n in hilang.most_common(10):
        print(f'    {n:4d}x  "{ka}"')
    print()

**Insight:**

Daftar substitusi kata biasanya memperlihatkan dua kelompok yang penanganannya
berbeda. Kelompok pertama adalah kata yang bunyinya memang mirip; kesalahan di
sini dapat diperbaiki tanpa melatih ulang model, yaitu dengan penskoran ulang
terhadap daftar produk — mekanisme yang sudah dipakai layanan REST FinSense.

Kelompok kedua adalah kata bilangan dan kata harga yang hilang atau tertukar.
Kesalahan ini berdampak langsung pada nominal transaksi, sehingga lebih berisiko
daripada salah eja nama produk, dan menjadi alasan aplikasi tetap meminta
konfirmasi kasir sebelum transaksi dicatat.

# **EVALUASI TAMBAHAN: KORPUS TAHAP 1 (MODEL YANG DILAYANKAN)**

Dua skenario di atas menilai model v11 yang dilatih di atas `finsense_expanded`.
Model yang sesungguhnya berjalan di aplikasi adalah hasil Tahap 2 (v12), dan ia
dilatih di atas korpus yang berbeda: potongan hasil *forced alignment* dari
rekaman yang sama, diperkaya augmentasi yang meniru kondisi warung dan rekaman
browser. Mengujinya dengan pembagian data di atas akan bocor, sebab korpus
latihnya diturunkan dari seluruh rekaman manusia yang ada.

Bagian ini menguji model tersebut pada himpunan yang memang disisihkan untuknya,
yaitu `manifest_val.csv` — klip yang **rekaman sumbernya** tidak pernah dipakai
melatih.

**Batas kesahihannya, dinyatakan terang-terangan:** pemisahan pada korpus itu
dilakukan per rekaman sumber, **bukan per penutur**. Seluruh 20 penutur tetap
muncul di sisi latih, jadi angka di bagian ini adalah *speaker-dependent* pada
rekaman baru — bukan pengganti skenario *speaker-independent*. Untuk klaim
*speaker-independent* atas v12, korpus Tahap 1 harus dibangun ulang dengan
penutur uji dikunci di luar.

Nilainya tetap besar untuk dua hal: ia satu-satunya angka CER/WER yang berlaku
bagi model yang benar-benar dilayankan, dan ia memecah hasil menurut **bentuk
ucapan** (produk saja, kalimat penuh, intent+produk, produk+jumlah) serta
**kondisi akustik** (bersih vs teraugmentasi) — persis dua hal yang membedakan
kondisi lapangan dari kondisi laboratorium.

In [ ]:
hasil_t1 = None
if not KORPUS_TAHAP1 or not MODEL_TAHAP1:
    print('[LEWATI] Evaluasi korpus Tahap 1: '
          f'korpus={KORPUS_TAHAP1 or "tidak ada"}, model={MODEL_TAHAP1 or "tidak ada"}')
else:
    manifest = pd.read_csv(os.path.join(KORPUS_TAHAP1, 'manifest_val.csv'))
    manifest['teks'] = manifest['transcript'].apply(bersihkan_teks)
    if MAKS_SAMPEL:
        manifest = manifest.sample(n=min(MAKS_SAMPEL, len(manifest)),
                                   random_state=SEED).reset_index(drop=True)

    print(f'Korpus Tahap 1 (validasi): {len(manifest):,} klip dari '
          f'{manifest["source"].nunique():,} rekaman sumber')
    print(f'Model: {os.path.basename(MODEL_TAHAP1)}')

    model_t1 = keras.models.load_model(MODEL_TAHAP1, compile=False)
    wav_dir = os.path.join(KORPUS_TAHAP1, 'wav')

    baris, t_mulai = [], time.time()
    for awal in range(0, len(manifest), BATCH_INFER):
        potong = manifest.iloc[awal:awal + BATCH_INFER]
        X = np.stack([ekstrak_logmel(os.path.join(wav_dir, fn))
                      for fn in potong['filename']])
        probs = model_t1.predict(X, verbose=0)
        for (_, r), hyp in zip(potong.iterrows(), greedy_decode(probs)):
            k = jarak_edit(r['teks'], hyp)
            w = jarak_edit(r['teks'].split(), hyp.split())
            baris.append({
                'filename': r['filename'],
                'skema': 'tahap1_val',
                'variant': r['variant'],
                'augment': r['augment'],
                'kondisi_akustik': 'bersih' if r['augment'] == 'clean' else 'teraugmentasi',
                'source': r['source'],
                # Kolom penutur mengikuti aturan yang sama: rekaman bernomor 1..1200
                # berasal dari 20 penutur, klip gen_* tidak punya penutur.
                'penutur': id_penutur(r['source']),
                'sintetis': not str(r['source'])[0].isdigit(),
                'bocor_varian': False,
                'teks_acuan': r['teks'],
                'teks_prediksi': hyp,
                'n_karakter': k['n'], 'sub_kar': k['sub'], 'hap_kar': k['hap'],
                'sis_kar': k['sis'], 'err_kar': k['total'],
                'n_kata': w['n'], 'sub_kata': w['sub'], 'hap_kata': w['hap'],
                'sis_kata': w['sis'], 'err_kata': w['total'],
                'cer_ucapan': k['total'] / max(k['n'], 1),
                'wer_ucapan': w['total'] / max(w['n'], 1),
                'tepat': r['teks'] == hyp,
            })
        selesai = min(awal + BATCH_INFER, len(manifest))
        if selesai % (BATCH_INFER * 10) == 0 or selesai == len(manifest):
            lewat = time.time() - t_mulai
            print(f'  {selesai:,}/{len(manifest):,} klip | {lewat:.0f}s '
                  f'({1000*lewat/selesai:.0f} ms/klip)')

    hasil_t1 = pd.DataFrame(baris)
    keluaran_t1 = os.path.join(KELUARAN, 'transkripsi_tahap1_val.csv')
    hasil_t1.to_csv(keluaran_t1, index=False)
    print(f'\n  tersimpan: {keluaran_t1}')

    def ringkas_t1(sub, label):
        return {
            'kelompok': label,
            'n_klip': len(sub),
            'CER_%': round(100 * sub.err_kar.sum() / max(sub.n_karakter.sum(), 1), 2),
            'WER_%': round(100 * sub.err_kata.sum() / max(sub.n_kata.sum(), 1), 2),
            'exact_match_%': round(100 * sub.tepat.mean(), 2),
        }

    # Pecahan menurut sumber suara wajib ada: klip sintetis jauh lebih mudah
    # bagi model, sehingga angka gabungan menjadi lebih baik daripada kinerja
    # pada suara manusia — dan itulah angka yang dialami pengguna.
    tabel_t1 = pd.DataFrame(
        [ringkas_t1(hasil_t1, 'SEMUA')]
        + [ringkas_t1(g, f'sumber: {"sintetis" if sin else "manusia"}')
           for sin, g in hasil_t1.groupby('sintetis')]
        + [ringkas_t1(g, f'varian: {v}') for v, g in hasil_t1.groupby('variant')]
        + [ringkas_t1(g, f'varian manusia: {v}')
           for v, g in hasil_t1[~hasil_t1.sintetis].groupby('variant')]
        + [ringkas_t1(g, f'akustik: {a}') for a, g in hasil_t1.groupby('kondisi_akustik')]
    )
    tabel_t1.to_csv(os.path.join(KELUARAN, 'cer_wer_tahap1_val.csv'), index=False)
    print()
    print(tabel_t1.to_string(index=False))

**Insight:**

Baris `varian: produk` adalah yang paling dekat dengan cara aplikasi dipakai:
kasir menyebut nama produk saja, tanpa kalimat pembungkus. Bandingkan dengan
baris `varian: full` — selisih keduanya menunjukkan apakah model masih
bergantung pada konteks kalimat untuk mengenali nama produk.

Baris `sumber: manusia` adalah angka yang pantas dikutip, bukan baris `SEMUA`.
Klip sintetis jauh lebih mudah bagi model, sehingga angka gabungan selalu
terlihat lebih baik daripada yang dialami pengguna sungguhan.

Baris `akustik: bersih` versus `akustik: teraugmentasi` perlu dibaca dengan
hati-hati. Model ini dilatih pada korpus yang 79% klipnya teraugmentasi, jadi
klip bersih justru menjadi bentuk yang lebih jarang dilihatnya. Bila klip
teraugmentasi ternyata tidak lebih buruk — atau bahkan sedikit lebih baik —
kesimpulannya bukan "augmentasi meningkatkan akurasi", melainkan bahwa dosis
augmentasi saat pelatihan sudah cukup tinggi sehingga kondisi bersih tidak lagi
menjadi acuan yang istimewa. Periksa arah selisihnya per varian sebelum menarik
kesimpulan apa pun; bila arahnya tidak konsisten antar varian, selisih itu
lebih mungkin derau sampel daripada efek nyata.

Berkas `transkripsi_tahap1_val.csv` memakai skema kolom yang sama dengan berkas
transkripsi skenario lain, sehingga notebook `08` dapat langsung memakainya
sebagai kondisi masukan tambahan — NLU diuji pada ucapan sepotong, bukan hanya
pada kalimat penuh.

# **LATENSI MODEL PER UCAPAN**

Tahap ini mengukur waktu yang dibutuhkan model untuk memproses satu ucapan:
ekstraksi fitur dan *forward pass*, satu per satu tanpa *batching*, seperti saat
melayani permintaan sungguhan.

Angka di sini adalah **latensi model**, bukan *response time* layanan REST.
*Response time* utuh — termasuk unggah audio, hop backend, dan penskoran ulang
leksikon — diukur di dalam layanan (`ai-service/serve_model.py`) dan di sisi
browser, lalu ditampilkan pada popup hasil.

In [ ]:
skema_latensi = skenario[0]
d = HASIL[skema_latensi]
model = MODEL[skema_latensi]
berkas_uji = d['filename'].head(N_LATENSI).tolist()

# Pemanasan: pemanggilan pertama memicu penelusuran graf, jangan ikut diukur.
_ = model.predict(ekstrak_logmel(os.path.join(AUDIO_DIR, berkas_uji[0]))[None, ...], verbose=0)

t_fitur, t_model = [], []
for fn in berkas_uji:
    t0 = time.perf_counter()
    fitur = ekstrak_logmel(os.path.join(AUDIO_DIR, fn))
    t1 = time.perf_counter()
    _ = model.predict(fitur[None, ...], verbose=0)
    t2 = time.perf_counter()
    t_fitur.append((t1 - t0) * 1000)
    t_model.append((t2 - t1) * 1000)

t_fitur, t_model = np.array(t_fitur), np.array(t_model)
t_total = t_fitur + t_model
tabel_latensi = pd.DataFrame([
    {'tahap': 'ekstraksi fitur', 'rerata_ms': t_fitur.mean(), 'median_ms': np.median(t_fitur),
     'p95_ms': np.percentile(t_fitur, 95)},
    {'tahap': 'inferensi model', 'rerata_ms': t_model.mean(), 'median_ms': np.median(t_model),
     'p95_ms': np.percentile(t_model, 95)},
    {'tahap': 'total', 'rerata_ms': t_total.mean(), 'median_ms': np.median(t_total),
     'p95_ms': np.percentile(t_total, 95)},
]).round(1)
tabel_latensi.to_csv(os.path.join(KELUARAN, 'latensi_model.csv'), index=False)
print(f'Diukur pada {len(berkas_uji)} ucapan, perangkat: '
      f'{"GPU" if tf.config.list_physical_devices("GPU") else "CPU"}')
tabel_latensi

**Insight:**

Pemisahan antara ekstraksi fitur dan inferensi model menentukan ke mana usaha
optimasi diarahkan. Bila ekstraksi fitur yang dominan, percepatan didapat dari
jalur audionya (menghindari berkas sementara, memangkas keheningan lebih awal),
bukan dari mengecilkan model.

Perlu dicatat bahwa `model.predict` memang membawa overhead pemanggilan Keras
per ucapan; pada layanan, model dibungkus `tf.function` ber-signature tetap
sehingga overhead itu hilang. Karena itu angka di sini adalah batas atas yang
konservatif, bukan yang terbaik yang bisa dicapai.

# **EKSPOR HASIL**

Tahap ini menuliskan seluruh tabel dan berkas ringkasan ke folder keluaran agar
dapat langsung dirujuk pada laporan maupun notebook berikutnya.

In [ ]:
ringkasan = {
    'dataset': TRANSCRIPTS,
    'jumlah_ucapan_dataset': int(len(df)),
    'jumlah_penutur': int(n_penutur),
    'checkpoint': {s: MODEL_SKEMA[s] for s in HASIL},
    'parameter_split': {
        'seed': SEED, 'test_frac': TEST_FRAC, 'val_frac': VAL_FRAC,
        'n_speakers': N_SPEAKERS, 'files_per_speaker': FILES_PER_SPK,
        'n_test_speakers': N_TEST_SPEAKERS, 'n_val_speakers': N_VAL_SPEAKERS,
    },
    'hasil': tabel_utama.to_dict(orient='records'),
    'latensi_model_ms': tabel_latensi.to_dict(orient='records'),
    'waktu_evaluasi': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(os.path.join(KELUARAN, 'ringkasan_evaluasi.json'), 'w', encoding='utf-8') as f:
    json.dump(ringkasan, f, indent=2, ensure_ascii=False)

# Tabel utama dalam format Markdown, siap disalin ke laporan.
with open(os.path.join(KELUARAN, 'tabel_cer_wer.md'), 'w', encoding='utf-8') as f:
    f.write(tabel_utama.to_markdown(index=False))

print('Berkas keluaran di', os.path.abspath(KELUARAN) + ':')
for nama in sorted(os.listdir(KELUARAN)):
    ukuran = os.path.getsize(os.path.join(KELUARAN, nama))
    print(f'  {nama:36s} {ukuran/1024:8.1f} KB')

# **KESIMPULAN**

In [ ]:
print('=' * 78)
print('RINGKASAN EVALUASI KEANDALAN MODEL CNN-BiGRU-CTC')
print('=' * 78)
for s in HASIL:
    d = HASIL[s]
    cer = 100 * d.err_kar.sum() / max(d.n_karakter.sum(), 1)
    wer = 100 * d.err_kata.sum() / max(d.n_kata.sum(), 1)
    print(f'\n{LABEL[s]}')
    print(f'  checkpoint    : {os.path.basename(MODEL_SKEMA[s])}')
    print(f'  ucapan uji    : {len(d):,} dari {d.loc[d.penutur>=0, "penutur"].nunique()} penutur')
    print(f'  CER           : {cer:.2f}%')
    print(f'  WER           : {wer:.2f}%')
    print(f'  exact match   : {100*d.tepat.mean():.2f}%')

if len(HASIL) == 2:
    ca = 100 * HASIL['speaker_dependent'].err_kar.sum() / max(HASIL['speaker_dependent'].n_karakter.sum(), 1)
    cb = 100 * HASIL['speaker_independent'].err_kar.sum() / max(HASIL['speaker_independent'].n_karakter.sum(), 1)
    print(f'\nSelisih CER (speaker-independent - speaker-dependent): {cb-ca:+.2f} poin persen')
    print(f'Rasio                                                : {cb/max(ca,1e-9):.2f}x')
else:
    print('\nCatatan: hanya satu skenario yang terevaluasi. Untuk perbandingan yang sah,')
    print('latih checkpoint skenario yang tersisa lebih dulu (lihat bagian Syarat menjalankan).')
if hasil_t1 is not None:
    cer_t1 = 100 * hasil_t1.err_kar.sum() / max(hasil_t1.n_karakter.sum(), 1)
    wer_t1 = 100 * hasil_t1.err_kata.sum() / max(hasil_t1.n_kata.sum(), 1)
    print(f'\nKorpus Tahap 1 (model yang dilayankan, {os.path.basename(MODEL_TAHAP1)})')
    print(f'  klip validasi : {len(hasil_t1):,} (rekaman sumber baru, penutur TIDAK baru)')
    print(f'  CER           : {cer_t1:.2f}%')
    print(f'  WER           : {wer_t1:.2f}%')
    print(f'  exact match   : {100*hasil_t1.tepat.mean():.2f}%')

berkas_siap = [f'transkripsi_{s}.csv' for s in HASIL]
if hasil_t1 is not None:
    berkas_siap.append('transkripsi_tahap1_val.csv')
print('\nBerkas per ucapan siap dipakai notebook 08:', ', '.join(berkas_siap))

**Insight akhir:**

Angka *speaker-dependent* menjawab pertanyaan "apakah arsitektur dan pelatihan
sudah benar" — model yang gagal pada penutur yang sudah dikenalnya berarti belum
belajar. Angka *speaker-independent* menjawab pertanyaan yang jauh lebih penting
bagi FinSense: apakah aplikasi ini akan bekerja untuk pedagang yang suaranya
belum pernah direkam sama sekali. Itulah angka yang pantas dijadikan dasar klaim
kelayakan sistem, dan itu pula yang menjadi acuan notebook `08` saat menilai
NLU di atas teks hasil transkripsi.

Bila selisih keduanya lebar, penyebab yang paling mungkin bukan arsitekturnya,
melainkan jumlah penutur pada data latih. Dua puluh penutur adalah jumlah yang
kecil untuk ASR; menambah penutur biasanya menaikkan kinerja lebih banyak
daripada menambah parameter model.